In [1]:
import os
import time
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient


user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_KEY")

login(token=hf_token)

In [3]:
class dtp_block_d(layers.Layer):
    def __init__(self, filters, kernel_size=4, strides=2):
        super(dtp_block_d, self).__init__(name='pc_block_d')
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        
        # Forward pass downsampling
        self.conv = layers.Conv2D(filters, kernel_size, strides=strides, padding='same')
        self.ln = layers.LayerNormalization()
        self.leaky = layers.LeakyReLU(0.2)
        
        # Upsampling geometry (no trainable weights, safe in __init__)
        self.upsample = layers.UpSampling2D(size=(strides, strides), interpolation='nearest')
        
        # Initialize empty, defined dynamically in build()
        self.up_conv = None

    def build(self, input_shape):
        # Dynamically extract the exact incoming channel count
        in_channels = input_shape[-1]
        
        # Create the reconstruction convolution to perfectly match the input depth
        self.up_conv = layers.Conv2D(in_channels, kernel_size=3, padding='same', trainable=False)
        super(dtp_block_d, self).build(input_shape)

    def call(self, x):
        # 1. Forward feature extraction
        features = self.leaky(self.ln(self.conv(x)))
        
        # 2. Local reconstruction (checkerboard-proof)
        reconstructed_grid = self.upsample(features)
        reconstruction = self.up_conv(reconstructed_grid)
        
        # 3. Predictive error calculation
        return features, reconstruction

In [4]:
class dtp_discriminator(models.Model):
    def __init__(self):
        super(dtp_discriminator, self).__init__(name='pc_discriminator')
        self.block1 = dtp_block_d(64)   # Output: 32x32x64
        self.block2 = dtp_block_d(128)  # Output: 16x16x128
        self.block3 = dtp_block_d(256)  # Output: 8x8x256
        self.block4 = dtp_block_d(512)  # Output: 4x4x512

        self.flatten = layers.Flatten() # Output: 8192
        self.final_dense = layers.Dense(1)

        # --- DENOISING FEATURE MATCHING (DFM) DECODER ---
        # This acts as the bridge between the GAN and Target Propagation.
        # It takes the deep features and reconstructs an idealized target image.
        self.denoiser = tf.keras.Sequential([
            layers.Dense(4 * 4 * 512, activation='relu'),
            layers.Dropout(0.3),
            layers.Reshape((4, 4, 512)),
            
            # 4x4 -> 8x8
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(256, 3, padding='same', activation='relu'),
            layers.Dropout(0.3),
            
            # 8x8 -> 16x16
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(128, 3, padding='same', activation='relu'),
            layers.Dropout(0.3),
            
            # 16x16 -> 32x32
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(64, 3, padding='same', activation='relu'),
            layers.Dropout(0.3),
            
            # 32x32 -> 64x64x3 (Image dimensions)
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(3, 3, padding='same', activation='tanh')
        ])

    def call(self, x, training=False, decode_features=False):
        h1, rec1 = self.block1(x)
        h2, rec2 = self.block2(h1)
        h3, rec3 = self.block3(h2)
        h4, rec4 = self.block4(h3)

        flat_features = self.flatten(h4)
        
        # Final adversarial score
        score = self.final_dense(flat_features)
        
        features = [h1, h2, h3, h4]
        reconstructions = [rec1, rec2, rec3, rec4]
        
        # If DFM is active, decode the features to output an idealized target image
        denoised_image = None
        if decode_features:
            denoised_image = self.denoiser(flat_features)
            return score, features, reconstructions, denoised_image
            
        return score, features, reconstructions

In [5]:
class dtp_block_g(layers.Layer):
    def __init__(self, filters, kernel_size=4, strides=2):
        super(dtp_block_g, self).__init__(name='pc_block_g')
        self.filters = filters
        self.kernel_size = kernel_size
        self.strides = strides
        
        # --- THE ARCHITECTURE UPGRADE ---
        # 1. Replaced Conv2DTranspose with pure geometric upsampling
        self.upsample = layers.UpSampling2D(size=(strides, strides), interpolation='nearest')
        
        # 2. Added standard Conv2D for smooth feature learning
        self.conv_up = layers.Conv2D(filters, 3 if kernel_size % 2 == 0 else kernel_size, padding='same', use_bias=False)
        
        self.ln = layers.LayerNormalization()
        self.relu = layers.ReLU()
        
        self.predict_back_v2 = None 

    def build(self, input_shape):
        in_channels = input_shape[-1]
        # [PHASE 6] Upgrade to non-linear inverse mapping
        self.predict_back_v2 = tf.keras.Sequential([
            layers.Conv2D(in_channels, self.kernel_size, strides=self.strides, padding='same', use_bias=False),
            layers.LayerNormalization(),
            layers.LeakyReLU(0.2),
            layers.Conv2D(in_channels, 3, padding='same', activation='linear')
        ])
        super(dtp_block_g, self).build(input_shape)

    def call(self, x, training=True):
        # 1. Scale up the grid safely without overlapping gradient issues
        upscaled = self.upsample(x)
        
        # 2. Learn and refine the features smoothly
        features = self.conv_up(upscaled)
        
        # 3. Apply normalization and activation
        h = self.relu(self.ln(features))
        
        # 4. The internal reflection pathway
        prediction = self.predict_back_v2(h)
        
        return h, prediction

In [6]:
class dtp_generator(models.Model):
    def __init__(self):
        super(dtp_generator, self).__init__(name='pc_generator')
        # Defining the hierarchy of DTP blocks 
        self.block1 = dtp_block_g(512, strides=4) 
        self.block2 = dtp_block_g(256)
        self.block3 = dtp_block_g(128)
        self.block4 = dtp_block_g(64)

        # Final upsampling without Conv2DTranspose checkerboarding
        self.final_layer = tf.keras.Sequential([
            layers.UpSampling2D(size=(2, 2), interpolation='nearest'),
            layers.Conv2D(
                3, 3, padding='same', 
                kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.02),
                use_bias=False, activation='tanh'
            )
        ])

    def call(self, x, training=True):
        h1, pred1 = self.block1(x, training=training)
        h2, pred2 = self.block2(h1, training=training)
        h3, pred3 = self.block3(h2, training=training)
        h4, pred4 = self.block4(h3, training=training)
        
        # This now safely upsamples and convolves without checkerboarding
        output = self.final_layer(h4)
        

        features = [h1, h2, h3, h4]
        predictions = [pred1, pred2, pred3, pred4]
        
        inputs = [x, h1, h2, h3]
        return output, features, predictions, inputs

In [7]:
# Re-build the "Body"
generator = dtp_generator()
discriminator = dtp_discriminator()

# Initialize the layers with dummy data to trigger the build() methods
noise_testparam = tf.random.normal([1, 1, 1, 100])
picture_testparam = tf.random.normal([1, 64, 64, 3])

_ = generator(noise_testparam, training=False)
_ = discriminator(picture_testparam)

print("Architecture built. Checking summaries...")
generator.summary()
discriminator.summary()

In [8]:
# 1. Define local path and parameters
# Note: Keras needs the path to the folder ABOVE the images folder 
# if your images are in /images/subfolder. If all images are in /images, 
# point to the parent of /images.
DATASET_PATH = "/kaggle/input/datasets/splcher/animefacedataset"
IMG_HEIGHT, IMG_WIDTH = 64, 64
BATCH_SIZE = 32

# 2. Load the dataset using Keras utilities
# label_mode=None ensures it only returns images, not labels
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    label_mode=None,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

# 3. Apply Normalization [-1, 1] for your DTP-GAN
# (x - 127.5) / 127.5 is standard for Tanh activation
train_ds = train_ds.map(lambda x: (x - 127.5) / 127.5)

# 4. Performance optimizations
# .cache() keeps images in RAM so they aren't re-read from disk every epoch
# .prefetch() overlaps data preprocessing with model execution
train_ds = train_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

print("Dataset successfully loaded for single-device training.")

In [9]:
for batch in train_ds.take(1):
    print("Batch shape:", batch.shape)
    print("Min value:", tf.reduce_min(batch).numpy())
    print("Max value:", tf.reduce_max(batch).numpy())

In [10]:
# 1. OPTIMIZERS
# Standard AdamW replaces the manual global unrolled loop
generator_optimizer = tf.keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=1e-4)
discriminator_optimizer = tf.keras.optimizers.AdamW(learning_rate=5e-6, weight_decay=1e-4)

# 2. SETUP CHECKPOINTS
checkpoint_dir = './training_checkpoints'
ckpt = tf.train.Checkpoint(generator=generator, discriminator=discriminator, g_opt=generator_optimizer, d_opt=discriminator_optimizer)
checkpoint_manager = tf.train.CheckpointManager(ckpt, checkpoint_dir, max_to_keep=3)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# --- DFM & TARGET PROPAGATION CONSTANTS ---
CLIP_NORM = tf.constant(1.0, dtype=tf.float32)     # Hard ceiling for gradient shocks
DFM_NOISE_STD = 0.1                                # Gaussian noise for Discriminator Denoiser
LAMBDA_DFM = tf.constant(1.0, dtype=tf.float32)   # [PHASE 6] Reduced to prevent whiteout collapse
ETA_TARGET = tf.constant(0.5, dtype=tf.float32)    # Target propagation step size
LAMBDA_INV = tf.constant(1.0, dtype=tf.float32)      # [PHASE 6] Increased for inverse mapping warm-up
DTP_NOISE_STD = 0.05                               # Gaussian noise for DTP inverse training

@tf.function 
def train_step(real_images):
    current_batch_size = tf.shape(real_images)[0]
    noise = tf.random.normal([current_batch_size, 1, 1, 100])

    with tf.GradientTape(persistent=True) as tape:
        
        # 1. GENERATOR FORWARD PASS
        generated_images, g_feats, g_preds, g_inputs = generator(noise, training=True) 
        h1, h2, h3, h4 = g_feats
        pred1, pred2, pred3, pred4 = g_preds
        in1, in2, in3, in4 = g_inputs
        
        # 2. DISCRIMINATOR DFM TRAINING (Denoiser Objective)
        noisy_real_images = real_images + tf.random.normal(shape=tf.shape(real_images), mean=0.0, stddev=DFM_NOISE_STD)
        noisy_real_images = tf.clip_by_value(noisy_real_images, -1.0, 1.0)
        
        # Use clean real images for the true adversarial signal
        real_output, _, _ = discriminator(real_images, training=True, decode_features=False)
        # Use noisy real images ONLY to train the denoiser
        _, _, _, denoised_real = discriminator(noisy_real_images, training=True, decode_features=True)
        
        fake_output, _, _, denoised_fake = discriminator(generated_images, training=True, decode_features=True)

        # 3. LOSS CALCULATIONS
        g_adv_loss = cross_entropy(tf.ones_like(fake_output), fake_output)
        d_adv_loss = cross_entropy(tf.ones_like(real_output) * 0.9, real_output) + \
                     cross_entropy(tf.zeros_like(fake_output), fake_output)

        dfm_loss = tf.reduce_mean(tf.square(real_images - denoised_real))
        total_disc_loss = d_adv_loss + (LAMBDA_DFM * dfm_loss)
        
        # 4. DECENTRALIZED TARGET PROPAGATION
        T_img = tf.clip_by_value(tf.stop_gradient(denoised_fake), -1.0, 1.0)
        
        # Top-level target loss
        target_pull_loss = tf.reduce_mean(tf.square(generated_images - T_img))
        loss_final = g_adv_loss + (LAMBDA_DFM * target_pull_loss)
        
        # Suspend recording to prevent memory bloat when not calculating second-order derivatives.
        with tape.stop_recording():
            grad_h4 = tape.gradient(loss_final, h4)
        
        T4 = tf.stop_gradient(h4 - ETA_TARGET * grad_h4)
        
        # Propagate targets backwards using frozen inverse mappings (Difference Target Propagation)
        T3 = tf.stop_gradient(generator.block4.predict_back_v2(T4) + h3 - pred4)
        T2 = tf.stop_gradient(generator.block3.predict_back_v2(T3) + h2 - pred3)
        T1 = tf.stop_gradient(generator.block2.predict_back_v2(T2) + h1 - pred2)
        
        # Local Losses for each block
        def calc_inv_loss(block, x_in, h_out):
            # Decouple inverse training from forward weights by treating them as constants.
            clean_in = tf.stop_gradient(x_in)
            clean_out = tf.stop_gradient(h_out)
            
            # [PHASE 6] Pure DTP: Remove random noise to learn exact inverse
            pred_x = block.predict_back_v2(clean_out)
            return tf.reduce_mean(tf.square(pred_x - clean_in))
            
        inv_loss_b4 = calc_inv_loss(generator.block4, in4, h4)
        inv_loss_b3 = calc_inv_loss(generator.block3, in3, h3)
        inv_loss_b2 = calc_inv_loss(generator.block2, in2, h2)
        inv_loss_b1 = calc_inv_loss(generator.block1, in1, h1)

        loss_b4 = tf.reduce_mean(tf.square(h4 - T4)) + LAMBDA_INV * inv_loss_b4
        loss_b3 = tf.reduce_mean(tf.square(h3 - T3)) + LAMBDA_INV * inv_loss_b3
        loss_b2 = tf.reduce_mean(tf.square(h2 - T2)) + LAMBDA_INV * inv_loss_b2
        loss_b1 = tf.reduce_mean(tf.square(h1 - T1)) + LAMBDA_INV * inv_loss_b1
        
        # Total generator loss (purely for logging progress, NOT for global backprop)
        total_gen_loss = loss_final + loss_b4 + loss_b3 + loss_b2 + loss_b1

    # 5. CALCULATE & APPLY GRADIENTS (Severed Graph / No Global Backprop)
    
    # Discriminator (Global update for disc only)
    disc_gradients = tape.gradient(total_disc_loss, discriminator.trainable_variables)
    
    # Generator Local Gradients
    grad_final = tape.gradient(loss_final, generator.final_layer.trainable_variables)
    grad_b4 = tape.gradient(loss_b4, generator.block4.trainable_variables)
    grad_b3 = tape.gradient(loss_b3, generator.block3.trainable_variables)
    grad_b2 = tape.gradient(loss_b2, generator.block2.trainable_variables)
    grad_b1 = tape.gradient(loss_b1, generator.block1.trainable_variables)
    
    del tape # Manually delete the persistent tape to free memory

    # Clip and apply discriminator gradients
    disc_gradients, _ = tf.clip_by_global_norm(disc_gradients, CLIP_NORM)
    discriminator_optimizer.apply_gradients(zip(disc_gradients, discriminator.trainable_variables))
    
    # Collect all generator gradients and variables into single lists
    all_gen_grads = grad_final + grad_b4 + grad_b3 + grad_b2 + grad_b1
    all_gen_vars = (generator.final_layer.trainable_variables + 
                    generator.block4.trainable_variables + 
                    generator.block3.trainable_variables + 
                    generator.block2.trainable_variables + 
                    generator.block1.trainable_variables)

    # Clip and apply generator gradients globally, but calculated locally
    all_gen_grads, _ = tf.clip_by_global_norm(all_gen_grads, CLIP_NORM)
    generator_optimizer.apply_gradients(zip(all_gen_grads, all_gen_vars))

    return total_gen_loss, total_disc_loss


In [11]:
TOTAL_PREVIOUS_EPOCHS = 0

In [ ]:
# 1. Set how long you want to train
EPOCHS = 10 

# 2. Create a static seed so we can see how the SAME images evolve
num_examples_to_generate = 16 # Generate a 4x4 grid
seed = tf.random.normal([num_examples_to_generate, 1, 1, 100])

def generate_and_save_images(model, epoch, test_input):
    # Catch all returns from the FW-DTP architecture
    predictions, _, _, _ = model(test_input, training=False)

    fig = plt.figure(figsize=(4, 4))

    for i in range(predictions.shape[0]):
        plt.subplot(4, 4, i+1)
        # Scale back from [-1, 1] to [0, 1] for matplotlib
        img = (predictions[i, :, :, :] + 1.0) / 2.0
        plt.imshow(img)
        plt.axis('off')

    plt.savefig(f'image_at_epoch_{epoch:04d}.png')
    # OPTIMIZATION 1: Explicitly close the 'fig' object. 
    plt.close(fig) 

def train(dataset, epochs):
    for epoch in range(epochs):
        start = time.time()
        
        # Container to record loss progress
        gen_loss_total = 0.0
        disc_loss_total = 0.0
        batch_count = 0

        actual_epoch = epoch + 1 + TOTAL_PREVIOUS_EPOCHS
        print(f"\nEpoch {actual_epoch} (FW-DTP & DFM) is running...")
        
        for image_batch in dataset:
            g_loss, d_loss = train_step(image_batch)
            
            # OPTIMIZATION 2: Cast the tensor losses to pure Python floats.
            gen_loss_total += float(g_loss)
            disc_loss_total += float(d_loss)
            batch_count += 1
            
        # Calculate average to track performance
        avg_g_loss = gen_loss_total / batch_count
        avg_d_loss = disc_loss_total / batch_count
        
        print(f"Finished in {time.time()-start:.2f} seconds")
        print(f"Gen Loss: {avg_g_loss:.4f} | Disc Loss: {avg_d_loss:.4f}")
        
        # Save the model
        checkpoint_manager.save() 
        
        # Pass actual_epoch so it saves sequentially (e.g., image_at_epoch_0061.png)
        generate_and_save_images(generator, actual_epoch, seed)

# GO! Start Training
train(train_ds, EPOCHS)

In [13]:
from huggingface_hub import HfApi, create_repo

# 1. Initialize the API with your HF token
# Make sure hf_token is defined or retrieved from Kaggle Secrets
api = HfApi(token=hf_token)

# 2. Define your repository ID
repo_id = "RinKana/DTP-GAN-V3-A"

# 3. Create the repository on Hugging Face (skips if already exists)
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repository {repo_id} is ready!")
except Exception as e:
    print(f"Failed to create repository: {e}")

# 4. Upload the training checkpoints folder
# This ensures all epoch weights are preserved
print("Uploading checkpoint folder to Hugging Face...")
api.upload_folder(
    folder_path="./training_checkpoints", 
    repo_id=repo_id,
    repo_type="model",
    path_in_repo="checkpoints" 
)

# 5. Optional: Upload the notebook or source code
# Crucial for reproducing the PC-block architecture later
# api.upload_file(
#     path_or_fileobj="your_notebook.ipynb", 
#     path_in_repo="notebook.ipynb", 
#     repo_id=repo_id
# )

print(f"Upload successful! Model is secured at: https://huggingface.co/{repo_id}")

In [14]:
from huggingface_hub import HfApi, create_repo

# 1. Initialize API (ensure hf_token is available in your environment)
api = HfApi(token=hf_token)

# 2. Define repository ID
repo_id = "RinKana/DTP-GAN-V2-A"

# 3. Create repo if it doesn't exist
try:
    create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    print(f"Repository {repo_id} is ready!")
except Exception as e:
    print(f"Repository status: {e}")

# 4. Upload images to 'picture/epoch_rt1'
print("Starting batch upload for epoch images...")

for i in range(1, 41):
    # Format the filename to match /kaggle/working/image_at_epoch_0001.png
    file_name = f"image_at_epoch_{i:04d}.png"
    local_path = f"/kaggle/working/{file_name}"
    
    if os.path.exists(local_path):
        api.upload_file(
            path_or_fileobj=local_path,
            path_in_repo=f"picture/epoch_1-40/{file_name}",
            repo_id=repo_id,
            repo_type="model"
        )
        print(f"Uploaded: {file_name}")
    else:
        print(f"Warning: {local_path} not found, skipping.")

print(f"Success! All images uploaded to: https://huggingface.co/{repo_id}/tree/main/picture/epoch_1-40")